In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle

In [2]:
sam = SAM()
sam.load_data('Active_SAM_joined/SAM_Allen_Insitute_NN_subclass_250.h5ad')

In [3]:
hypo = ['15 HY Gnrh1 Glut','16 HY MM Glut','14 HY Glut','13 CNU-HYa Glut','12 HY GABA','11 CNU-HYa GABA']
t0 = time.time()
a = 0
parent_dict = {}
for item in sam.adata:
    a += 1
    parent_dict[list(item.obs['cluster_alias'])[0]] = list(item.obs['supertype_id_label'])[0]
    parent_dict[list(item.obs['supertype_id_label'])[0]] = list(item.obs['subclass_id_label'])[0]
    parent_dict[list(item.obs['subclass_id_label'])[0]] = list(item.obs['class_id_label'])[0]
    if list(item.obs['class_id_label'])[0] in hypo:
        parent_dict[list(item.obs['class_id_label'])[0]] = 'hypo'
    else:
        parent_dict[list(item.obs['class_id_label'])[0]] = 'not hypo'
    if a % 1000 == 0:
        t1 = time.time()
        print(str(a/len(sam.adata)) + ' percent in ' + str(t1-t0) + ' seconds')
parent_dict['Unlabeled'] = 'Unlabeled'

0.012790667928679236 percent in 8.59237289428711 seconds
0.025581335857358473 percent in 17.133203506469727 seconds
0.03837200378603771 percent in 25.705772638320923 seconds
0.051162671714716945 percent in 34.31266164779663 seconds
0.06395333964339618 percent in 42.93346047401428 seconds
0.07674400757207542 percent in 51.59315013885498 seconds
0.08953467550075465 percent in 60.25353527069092 seconds
0.10232534342943389 percent in 68.89779710769653 seconds
0.11511601135811313 percent in 77.77915859222412 seconds
0.12790667928679236 percent in 86.33808779716492 seconds
0.14069734721547159 percent in 94.9493420124054 seconds
0.15348801514415084 percent in 103.53034710884094 seconds
0.16627868307283006 percent in 112.21834254264832 seconds
0.1790693510015093 percent in 120.87559008598328 seconds
0.19186001893018853 percent in 129.54204273223877 seconds
0.20465068685886778 percent in 138.2368597984314 seconds
0.217441354787547 percent in 146.9080798625946 seconds
0.23023202271622625 percent

In [4]:
import pickle

with open('parent_dict.pkl', 'wb') as f:
    pickle.dump(parent_dict, f)